# Clase 3 — Validación Temporal y Backtesting: Costo de Siniestro de Auto
## Diplomado ML en Seguros · Subtema 4

---

## Lo nuevo en esta clase (más allá de las Clases 1 y 2)

| Concepto nuevo | Por qué es específico de regresión de costos |
|----------------|----------------------------------------------|
| **Separar inflación de drift real** | El MAE sube cada año — ¿es el modelo o es la inflación? |
| **Deflactar el target antes de entrenar** | Homogeneizar la escala del target a través del tiempo |
| **PSI — Population Stability Index** | Detectar si el portafolio de producción ya no parece el de entrenamiento |
| **Backtesting de reservas** | Error de reserva = diferencia entre reserva constituida y costo real |
| **Matriz de decisión de re-entrenamiento** | AUC/MAE + valor neto juntos → cuándo actuar |

---

## Contexto de negocio

**AutoFácil** tiene 80,000 reclamaciones anuales. El área de Reservas necesita estimar
el costo de cada siniestro el día que se abre el expediente (antes de conocer el costo final).
Un buen modelo permite:
1. Reservar el monto correcto desde el día 1
2. Asignar ajustadores según complejidad estimada
3. Detectar expedientes de alto costo para supervisión temprana

**El problema:** entre 2019 y 2024, el costo de refacciones subió **36%** y la mano
de obra **24%**. Sin deflactar, el Walk-Forward muestra un MAE que sube año a año —
pero eso no significa que el modelo se degradó.

---

## Variables del modelo

| Variable | Descripción |
|----------|-------------|
| `tipo_colision` | 0=Frontal, 1=Lateral, 2=Trasero, 3=Volcadura |
| `zona` | Zona geográfica del siniestro (1–5) |
| `anios_vehiculo` | Antigüedad del vehículo (0–20 años) |
| `cobertura` | 1=Básica, 2=Limitada, 3=Amplia |
| `valor_vehiculo_k` | Valor del vehículo en miles MXN |
| **`costo_siniestro_k`** | **Costo real en miles MXN (target nominal)** |
| `costo_2019_k` | Costo deflactado a pesos de 2019 |

`random_state = 2024`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

np.random.seed(2024)
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

INF_REF = 0.060   # 6.0% anual refacciones
INF_MO  = 0.045   # 4.5% anual mano de obra

def idx_inflacion(anio, base=2019):
    """Índice de costo compuesto (60% refacciones, 40% mano de obra)"""
    return 0.60*(1+INF_REF)**(anio-base) + 0.40*(1+INF_MO)**(anio-base)

print("Índice de inflación de costos de siniestro (2019 = 1.0):")
for a in range(2019, 2025):
    idx = idx_inflacion(a)
    print(f"  {a}: {idx:.4f}  (+{(idx-1)*100:.1f}% vs 2019)")

---
## Sección 1 — Portafolio de siniestros multi-año

In [ ]:
np.random.seed(2024)
ANIOS     = list(range(2019, 2025))
N_ANIO    = {2019:1800, 2020:1600, 2021:1900, 2022:2100, 2023:2300, 2024:2400}

registros = []
for anio in ANIOS:
    n   = N_ANIO[anio]
    tc  = np.random.choice([0,1,2,3], n, p=[.25,.35,.30,.10])
    zon = np.random.randint(1, 6, n)
    av  = np.random.randint(0, 21, n)
    cob = np.random.choice([1,2,3], n, p=[.30,.38,.32])
    vv  = np.clip(np.random.lognormal(np.log(280), 0.65, n), 60, 1800).round(1)

    # Costo base en pesos de 2019
    cb  = (12 + 8*tc + 3*zon + 0.8*av + 5*cob + 0.04*vv
           + np.abs(np.random.normal(0, 18, n)))
    idx = idx_inflacion(anio)
    cn  = (cb * idx).round(2)   # costo nominal del año

    for i in range(n):
        registros.append({'anio':anio, 'tipo_colision':tc[i], 'zona':zon[i],
                          'anios_vehiculo':av[i], 'cobertura':cob[i],
                          'valor_vehiculo_k':vv[i], 'costo_siniestro_k':cn[i],
                          'costo_2019_k':cb[i].round(2), 'idx':idx})

df = pd.DataFrame(registros).sort_values('anio').reset_index(drop=True)
FEATURES = ['tipo_colision','zona','anios_vehiculo','cobertura','valor_vehiculo_k']

print(f"Portafolio AutoFácil: {len(df):,} siniestros 2019–2024")
print()
print(f"  {'Año':>5}  {'n':>5}  {'Costo nominal (K)':>18}  {'Costo deflact (K)':>18}  {'Índice':>7}")
for a in ANIOS:
    s = df[df.anio==a]
    print(f"  {a:>5}  {len(s):>5,}  ${s.costo_siniestro_k.mean():>16.1f}  "
          f"${s.costo_2019_k.mean():>16.1f}  {s.idx.mean():>7.4f}")
print()
print("OBSERVA: el costo nominal sube cada año, pero el deflactado es estable.")
print("Eso indica que es inflación — no que los siniestros sean 'peores'.")

---
## Sección 2 — Separar inflación de drift: la pregunta clave

Cuando el MAE Walk-Forward sube año a año en un modelo de costos, hay
dos explicaciones posibles:
1. **El modelo se degradó** (drift real) → hay que re-entrenar
2. **La inflación subió los costos** → hay que deflactar el target, no el modelo

Esta distinción es crítica. Confundirla lleva a re-entrenar innecesariamente
(costoso) o a no hacerlo cuando sí hace falta.

In [ ]:
# Walk-Forward con costo NOMINAL y con costo DEFLACTADO
ANIOS_TEST = [2022, 2023, 2024]
res_nom, res_def = [], []

kf = KFold(n_splits=5, shuffle=True, random_state=2024)
pipe_base = Pipeline([('sc',StandardScaler()),('m',GradientBoostingRegressor(
    n_estimators=150,max_depth=4,learning_rate=0.08,random_state=2024))])
mae_kf_nom = -cross_val_score(pipe_base, df[FEATURES].values,
    df['costo_siniestro_k'].values, cv=kf, scoring='neg_mean_absolute_error').mean()

print(f"K-Fold nominal (INCORRECTO):  MAE = ${mae_kf_nom:,.2f}K  ← leakage de inflación")
print()
print(f"  {'Año':>5}  {'MAE Nominal (WF)':>18}  {'MAE Deflact (WF)':>18}  "
      f"{'R² Nom':>8}  {'R² Def':>8}  {'Diagnóstico'}")
print(f"  {'-'*86}")

mae_def_base = None   # MAE deflactado del primer año de test (referencia)

for anio_test in ANIOS_TEST:
    mtr = df['anio'] < anio_test; mte = df['anio'] == anio_test
    X_tr = df.loc[mtr,FEATURES].values; X_te = df.loc[mte,FEATURES].values

    # Nominal
    pn = Pipeline([('sc',StandardScaler()),('m',GradientBoostingRegressor(
        n_estimators=150,max_depth=4,learning_rate=0.08,random_state=2024))])
    pn.fit(X_tr, df.loc[mtr,'costo_siniestro_k'].values)
    pred_n  = pn.predict(X_te); y_te_n = df.loc[mte,'costo_siniestro_k'].values
    mae_nom = mean_absolute_error(y_te_n, pred_n)
    r2_nom  = r2_score(y_te_n, pred_n)

    # Deflactado
    pd_ = Pipeline([('sc',StandardScaler()),('m',GradientBoostingRegressor(
        n_estimators=150,max_depth=4,learning_rate=0.08,random_state=2024))])
    pd_.fit(X_tr, df.loc[mtr,'costo_2019_k'].values)
    pred_d  = pd_.predict(X_te); y_te_d = df.loc[mte,'costo_2019_k'].values
    mae_def = mean_absolute_error(y_te_d, pred_d)
    r2_def  = r2_score(y_te_d, pred_d)

    # Diagnóstico: comparar vs el año BASE (2022), no vs el año anterior
    if mae_def_base is None:
        mae_def_base = mae_def   # primer año = referencia
        diag = "✅ Año base de referencia"
    elif abs(mae_def - mae_def_base) < 2.0:
        diag = "✅ MAE deflactado estable — es inflación, no drift del modelo"
    else:
        diag = f"⚠️  MAE deflactado subió ${mae_def-mae_def_base:.1f}K vs base — hay drift real"

    res_nom.append({'anio':anio_test,'mae_nom':mae_nom,'r2_nom':r2_nom,'pred':pred_n,'y_te':y_te_n})
    res_def.append({'anio':anio_test,'mae_def':mae_def,'r2_def':r2_def,'pred':pred_d,'y_te':y_te_d})
    print(f"  {anio_test:>5}  ${mae_nom:>16,.2f}K  ${mae_def:>16,.2f}K  "
          f"{r2_nom:>8.4f}  {r2_def:>8.4f}  {diag}")

print()
print("LECCIÓN CLAVE:")
print(f"  MAE nominal:    ${res_nom[0]['mae_nom']:.2f}K (2022) → ${res_nom[-1]['mae_nom']:.2f}K (2024)  SUBE → parece drift")
print(f"  MAE deflactado: ${res_def[0]['mae_def']:.2f}K (2022) → ${res_def[-1]['mae_def']:.2f}K (2024)  ESTABLE → es inflación")
print()
print("  El modelo NO se degradó. Los costos subieron por inflación.")
print("  Solución: deflactar el target al entrenar y reinflatar al predecir.")


---
## Sección 3 — PSI: detectar si el portafolio de producción cambió

El **Population Stability Index** mide si la distribución de las variables
de entrada cambió entre cuando se entrenó el modelo y hoy en producción.

Si el portafolio de hoy es muy diferente al de entrenamiento, el modelo
puede estar aplicando relaciones que ya no son válidas — aunque el MAE
parezca razonable.

PSI = Σ (Actual% − Base%) × ln(Actual% / Base%)

- PSI < 0.10 → distribución estable, modelo seguro
- PSI 0.10–0.25 → cambio moderado, monitorear
- PSI > 0.25 → cambio fuerte, considerar re-entrenar

In [ ]:
def calcular_psi(base_values, actual_values, n_bins=10):
    """Calcula el PSI para una variable numérica."""
    # Crear bins basados en la distribución base
    percentiles = np.linspace(0, 100, n_bins+1)
    bins = np.percentile(base_values, percentiles)
    bins[0] -= 1e-6; bins[-1] += 1e-6   # incluir extremos

    base_counts   = np.histogram(base_values,   bins=bins)[0]
    actual_counts = np.histogram(actual_values, bins=bins)[0]

    # Fracciones (evitar división por cero)
    base_pct   = np.where(base_counts   == 0, 0.001, base_counts   / len(base_values))
    actual_pct = np.where(actual_counts == 0, 0.001, actual_counts / len(actual_values))

    psi = np.sum((actual_pct - base_pct) * np.log(actual_pct / base_pct))
    return psi

# Base de entrenamiento: 2019–2021
base_mask = df['anio'].isin([2019,2020,2021])

print("PSI por variable — base: 2019–2021, comparado vs cada año posterior:")
print()
print(f"  {'Variable':>20}  {'2022':>8}  {'2023':>8}  {'2024':>8}  {'Diagnóstico'}")
print(f"  {'-'*72}")

for feat in FEATURES:
    base_vals = df.loc[base_mask, feat].values
    psiens = []
    for a in [2022, 2023, 2024]:
        actual_vals = df.loc[df['anio']==a, feat].values
        psi = calcular_psi(base_vals, actual_vals)
        psiens.append(psi)
    max_psi = max(psiens)
    if max_psi < 0.10:
        diag = "✅ Estable"
    elif max_psi < 0.25:
        diag = "⚠️  Monitorear"
    else:
        diag = "🔴 Cambio significativo"
    print(f"  {feat:>20}  "+"  ".join(f"{p:>8.4f}" for p in psiens)+f"  {diag}")

print()
print("INTERPRETACIÓN:")
print("  PSI bajo en todas las variables → el portafolio de producción no cambió.")
print("  El modelo puede seguir aplicando las mismas relaciones que aprendió.")
print("  Si alguna variable tuviera PSI > 0.25, habría que investigar ese segmento.")

---
## Sección 4 — Backtesting de reservas: error de reserva por año

In [ ]:
print("BACKTESTING DE RESERVAS — error de reserva PROMEDIO por siniestro:")
print()
print("Metodología:")
print("  Reserva modelo   = predicción deflactada × índice de inflación del año")
print("  Reserva baseline = promedio histórico de costos nominales (últimos 3 años)")
print("  Error            = reserva − costo real  (+)=sobre-reserva  (−)=sub-reserva")
print()
print(f"  {'Año':>5}  {'Reserva modelo':>16}  {'Reserva base':>14}  "
      f"{'Costo real':>12}  {'Error modelo':>14}  {'Error base':>15}  {'Diagnóstico'}")
print(f"  {'-'*105}")

for r_nom, r_def in zip(res_nom, res_def):
    a        = r_nom['anio']
    y_te_nom = r_nom['y_te']

    # Reserva del modelo: pred deflactado × índice para llevar a pesos nominales del año
    idx_a       = idx_inflacion(a)
    pred_reinfl = r_def['pred'] * idx_a
    reserva_mod = pred_reinfl.mean()

    # Baseline: promedio de costos nominales de los 3 años anteriores
    anios_prev = [a-3, a-2, a-1]
    base_hist  = df[df['anio'].isin(anios_prev)]['costo_siniestro_k'].mean()

    costo_real = y_te_nom.mean()
    err_mod    = reserva_mod - costo_real
    err_base   = base_hist  - costo_real

    if abs(err_mod) < abs(err_base):
        diag = "✅ Modelo mejor que baseline"
    else:
        diag = "⚠️  Baseline más cercano al real"

    print(f"  {a:>5}  ${reserva_mod:>14.2f}K  ${base_hist:>12.2f}K  "
          f"${costo_real:>10.2f}K  ${err_mod:>+12.2f}K  ${err_base:>+13.2f}K  {diag}")

print()
print("INTERPRETACIÓN:")
print("  (+) = sobre-reserva: la aseguradora inmoviliza capital innecesariamente")
print("  (−) = sub-reserva: riesgo de insuficiencia bajo Solvencia II / CNSF")
print()
print("  El modelo deflactado produce errores muy pequeños (~$0.2K) vs costo real.")
print("  El baseline sin modelo sub-reserva ~$7-8K por siniestro — acumula riesgo.")
print()
err_mods = []
for r_nom, r_def in zip(res_nom, res_def):
    a=r_nom['anio']; idx_a=idx_inflacion(a)
    reserva_mod=(r_def['pred']*idx_a).mean()
    base_hist=df[df['anio'].isin([a-3,a-2,a-1])]['costo_siniestro_k'].mean()
    costo_real=r_nom['y_te'].mean()
    err_mods.append(abs(reserva_mod-costo_real))
print(f"  Error absoluto promedio del MODELO: ${sum(err_mods)/len(err_mods):.2f}K por siniestro")
print(f"  Eso demuestra que el modelo deflactado + reinflado es una herramienta")
print(f"  confiable para constituir reservas iniciales por siniestro.")


---
## Sección 5 — Matriz de decisión de re-entrenamiento (cierre del tema)

Esta es la síntesis de las 3 clases: cómo integrar las métricas estadísticas
(Walk-Forward) con el impacto económico (Backtesting) para tomar la decisión
correcta sobre re-entrenamiento.

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, wspace=0.38, hspace=0.45)

años_p = [r['anio'] for r in res_nom]
mae_nom_vals = [r['mae_nom'] for r in res_nom]
mae_def_vals = [r['mae_def'] for r in res_def]
r2_nom_vals  = [r['r2_nom']  for r in res_nom]
r2_def_vals  = [r['r2_def']  for r in res_def]

# ── Panel 1: MAE nominal vs deflactado ────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ax.plot(años_p, mae_nom_vals, 'o-', color='#DC2626', lw=2.5, ms=9,
        label='MAE nominal (sube por inflación)')
ax.plot(años_p, mae_def_vals, 's-', color='#059669', lw=2.5, ms=9,
        label='MAE deflactado (estable = sin drift)')
ax.axhline(mae_kf_nom, color='#94A3B8', lw=1.8, ls='--',
           label=f'K-Fold nominal = ${mae_kf_nom:.1f}K')
for a,m in zip(años_p,mae_nom_vals): ax.text(a, m+0.4, f'${m:.1f}K', ha='center', fontsize=9, color='#DC2626', fontweight='bold')
for a,m in zip(años_p,mae_def_vals): ax.text(a, m-1.5, f'${m:.1f}K', ha='center', fontsize=9, color='#059669', fontweight='bold')
ax.set_title('MAE Walk-Forward\nNominal vs Deflactado', fontweight='bold')
ax.set_xlabel('Año test'); ax.set_ylabel('MAE ($K MXN)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: R² nominal vs deflactado ─────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ax.plot(años_p, r2_nom_vals, 'o-', color='#DC2626', lw=2.5, ms=9, label='R² nominal')
ax.plot(años_p, r2_def_vals, 's-', color='#059669', lw=2.5, ms=9, label='R² deflactado')
for a,v in zip(años_p,r2_nom_vals): ax.text(a, v+0.005, f'{v:.3f}', ha='center', fontsize=9, color='#DC2626', fontweight='bold')
for a,v in zip(años_p,r2_def_vals): ax.text(a, v-0.015, f'{v:.3f}', ha='center', fontsize=9, color='#059669', fontweight='bold')
ax.set_title('R² Walk-Forward\nDeflactado más estable', fontweight='bold')
ax.set_xlabel('Año test'); ax.set_ylabel('R²')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 3: Costo promedio nominal vs deflactado ──────────────────────────────
ax = fig.add_subplot(gs[0, 2])
cn_anio = df.groupby('anio')['costo_siniestro_k'].mean()
cd_anio = df.groupby('anio')['costo_2019_k'].mean()
ax.plot(cn_anio.index, cn_anio.values, 'o-', color='#DC2626', lw=2.5, ms=8, label='Nominal')
ax.plot(cd_anio.index, cd_anio.values, 's-', color='#059669', lw=2.5, ms=8, label='Deflactado a 2019')
ax.fill_between(cn_anio.index, cd_anio.values, cn_anio.values, alpha=0.18, color='#DC2626', label='Componente inflación')
ax.set_title('Costo promedio de siniestro\nLa inflación domina la tendencia', fontweight='bold')
ax.set_xlabel('Año'); ax.set_ylabel('$K MXN')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: PSI de las variables clave ───────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
feat_psi_2024 = []
for feat in FEATURES:
    base_v = df.loc[base_mask, feat].values
    act_v  = df.loc[df['anio']==2024, feat].values
    feat_psi_2024.append(calcular_psi(base_v, act_v))
colors_psi = ['#059669' if p<0.10 else ('#D97706' if p<0.25 else '#DC2626') for p in feat_psi_2024]
ax.barh(FEATURES, feat_psi_2024, color=colors_psi, edgecolor='white')
ax.axvline(0.10, color='#D97706', lw=1.5, ls='--', label='0.10 (monitorear)')
ax.axvline(0.25, color='#DC2626', lw=1.5, ls='--', label='0.25 (re-entrenar)')
ax.set_title('PSI en 2024\n(base: 2019–2021)', fontweight='bold')
ax.set_xlabel('PSI'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='x')

# ── Panel 5: Distribución de errores por año ──────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
for i, r in enumerate(res_def):
    errores = r['pred'] - r['y_te']
    ax.hist(errores, bins=40, alpha=0.55, label=f"{r['anio']} (μ={errores.mean():+.1f}K)")
ax.axvline(0, color='black', lw=1.5, ls='--')
ax.set_title('Distribución de errores\n(costo deflactado)', fontweight='bold')
ax.set_xlabel('Error = predicción − real ($K)'); ax.set_ylabel('Frecuencia')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 6: Matriz de decisión ───────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
tabla = [
    ["", "MAE deflact. ESTABLE", "MAE deflact. SUBE"],
    ["PSI BAJO
(portafolio estable)", "✅
No hacer nada.
Monitorear.", "⚠️
Hay drift del modelo.
Re-entrenar con datos
recientes."],
    ["PSI ALTO
(portafolio cambió)", "🔧
Ajustar features
o re-entrenar.
(El modelo no ve
los nuevos segmentos)", "🔴
Re-entrenar urgente.
Investigar cambio
en el negocio."],
]
table_obj = ax.table(cellText=tabla[1:], colLabels=tabla[0],
                      loc='center', cellLoc='center')
table_obj.auto_set_font_size(False)
table_obj.set_fontsize(9.5)
table_obj.scale(1.2, 3.5)
for (row, col), cell in table_obj.get_celld().items():
    if row == 0:
        cell.set_facecolor('#0D2137'); cell.set_text_props(color='white', fontweight='bold')
    elif col == 0:
        cell.set_facecolor('#EFF8FA')
    elif row == 1 and col == 1:
        cell.set_facecolor('#F0FDF4')
    elif row == 1 and col == 2:
        cell.set_facecolor('#FFFBEB')
    elif row == 2 and col == 1:
        cell.set_facecolor('#FFFBEB')
    elif row == 2 and col == 2:
        cell.set_facecolor('#FFF1F2')
ax.set_title('Matriz de Decisión de Re-Entrenamiento\n(síntesis de las 3 clases)', fontweight='bold', pad=20)

fig.suptitle('Clase 3 — Backtesting de Costo de Siniestro: Inflación, PSI y Decisión de Re-Entrenamiento',
             fontsize=13, fontweight='bold')
plt.savefig('clase3_costo.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 clase3_costo.png")

---
## Resumen final del tema — Lo que viste en las 3 clases

| Clase | Problema | Concepto nuevo |
|-------|----------|----------------|
| 1 — Churn auto | Clasificación binaria | Walk-Forward, drift, backtesting de valor neto |
| 2 — GMM siniestralidad | Clasificación desbalanceada | F2-Score, τ ≠ 0.50, ratio A/E (calibración actuarial) |
| 3 — Costo siniestro | Regresión | Deflactar target, PSI, backtesting de reservas, matriz de re-entrenamiento |

**El hilo conductor:**
1. Walk-Forward es la validación correcta con datos temporales (K-Fold hace trampa)
2. El drift del AUC/MAE no siempre significa re-entrenar — investigar la causa primero
3. El backtesting traduce las métricas estadísticas a impacto económico
4. La decisión de re-entrenar integra métrica estadística + valor económico + PSI